In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper ,ArxivAPIWrapper
from langgraph.graph import StateGraph, END

In [12]:
load_dotenv(override=True)

llm = AzureChatOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0.7
)

response = llm.invoke("Explain what LangSmith does in simple terms.")
print(response.content)

LangSmith is a tool that helps developers build and improve applications that use language models (like AI that understands and generates text). It makes it easier to test, track, and debug how these language models work inside apps, so developers can create better, more reliable AI-powered software.


In [ ]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)

def arxiv_tool_function(query:str)->str:
    print("Using Tool Arxiv")
    arxiv_runner = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
    return arxiv_runner.invoke(query)
    
arxiv = Tool(
    name="arxiv",
    description = "Searches Academic paper on Arxiv related to a company or topic.",
    func= arxiv_tool_function
)

In [ ]:
def tavily_tool_func(query:str)->str:
    print("Using Tool Tavily")
    tavily_runner = TavilySearchResults(api_key = os.getenv("TAVILY_API_KEY"))
    return tavily_runner.invoke(query)

tavily = Tool(
    name ="tavily_search_tool",
    description="Perform a live web search for the latest financial and market information.",
    func=tavily_tool_func
)

In [ ]:
api_wrapper_wikipedia = WikipediaAPIWrapper(top_k_results=2,doc_content_chars_max=500)
def wikipedia_tool_func(query:str)->str:
    print("using Wikipedia Tool")
    wikipedia_runner = WikipediaQueryRun(api_wrapper=api_wrapper_wikipedia)
    return wikipedia_runner.run(query)

wikipedia = Tool(
    name="wikipedia",
    description="Fetches background information about a company or topic from wikipedia",
    func=wikipedia_tool_func
)